In [ ]:
!pip -q install rasterio 2>/dev/null
import torch
import numpy as np
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
DEV = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from google.colab import files
import os
import glob
files.upload()
for f in list(glob.glob("*(1)*")):
    os.rename(f, f.replace(" (1)", "").replace("(1)", ""))
print("files:", sorted(glob.glob("*.tif")))

In [ ]:
import rasterio
from rasterio.warp import transform as warp_transform
from scipy.ndimage import distance_transform_edt, uniform_filter

YEARS = list(range(2017, 2025))
CH_NAMES = ["built", "built_density", "prox_built", "prox_road", "prox_airport", "ndvi"]
USE_NDVI = all(os.path.exists(f"stack_{y}.tif") for y in (2017, 2019, 2024))
if not USE_NDVI:
    CH_NAMES = CH_NAMES[:-1]
AIRPORT_LONLAT = (77.7066, 13.1986)
STACK_YEARS = [2017, 2019, 2024]
REF = "pred_2024.tif"

with rasterio.open(REF) as s:
    TRANSFORM, CRS, H, W = s.transform, s.crs, s.height, s.width
print("grid", W, "x", H, CRS, "| channels", CH_NAMES)


def align(path, band=1):
    with rasterio.open(path) as s:
        a = s.read(band).astype("float32")
        nodata = s.nodata
    a[~np.isfinite(a)] = np.nan
    a[np.abs(a) > 1e30] = np.nan
    if nodata is not None:
        a[a == np.float32(nodata)] = np.nan
    out = np.full((H, W), np.nan, "float32")
    h, w = min(H, a.shape[0]), min(W, a.shape[1])
    out[:h, :w] = a[:h, :w]
    return out


def proximity(mask, scale):
    return np.exp(-distance_transform_edt(~mask) / scale).astype("float32")


xs, ys = warp_transform("EPSG:4326", CRS, [AIRPORT_LONLAT[0]], [AIRPORT_LONLAT[1]])
col, row = ~TRANSFORM * (xs[0], ys[0])
AIRPORT = (int(round(row)), int(round(col)))
airport_mask = np.zeros((H, W), bool)
if 0 <= AIRPORT[0] < H and 0 <= AIRPORT[1] < W:
    airport_mask[AIRPORT] = True
PROX_AIR = proximity(airport_mask, 300)


def nearest(year, options):
    return min(options, key=lambda o: abs(o - year))


def built_of(year):
    return np.clip(np.nan_to_num(align(f"dw_built_{year}.tif")), 0, 1)


def channels_from_built(built, year):
    density = uniform_filter(built, size=21, mode="nearest")
    prox_built = proximity(built > 0.5, 30)
    road = np.nan_to_num(align("pred_2017.tif" if year <= 2020 else "pred_2024.tif")) > 0.5
    prox_road = proximity(road, 30)
    channels = [built, density, prox_built, prox_road, PROX_AIR]
    if USE_NDVI:
        channels.append(np.nan_to_num((align(f"stack_{nearest(year, STACK_YEARS)}.tif", 6) + 1) / 2))
    return np.stack(channels).astype("float32")


series = np.stack([channels_from_built(built_of(y), y) for y in YEARS])
C = series.shape[1]
print("series", series.shape)

In [ ]:
import torch.nn as nn


class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hid, k=3):
        super().__init__()
        self.hid = hid
        self.conv = nn.Conv2d(in_ch + hid, 4 * hid, k, padding=k // 2)

    def forward(self, x, h, c):
        i, f, o, g = torch.chunk(self.conv(torch.cat([x, h], 1)), 4, 1)
        i, f, o, g = i.sigmoid(), f.sigmoid(), o.sigmoid(), g.tanh()
        c = f * c + i * g
        return o * c.tanh(), c


class ConvLSTMForecaster(nn.Module):
    def __init__(self, in_ch, hids=(48, 64)):
        super().__init__()
        self.cells = nn.ModuleList()
        prev = in_ch
        for hid in hids:
            self.cells.append(ConvLSTMCell(prev, hid))
            prev = hid
        self.head = nn.Conv2d(prev, 1, 1)

    def forward(self, seq):
        batch, steps, _, h, w = seq.shape
        hs = [torch.zeros(batch, cell.hid, h, w, device=seq.device) for cell in self.cells]
        cs = [torch.zeros(batch, cell.hid, h, w, device=seq.device) for cell in self.cells]
        for t in range(steps):
            x = seq[:, t]
            for i, cell in enumerate(self.cells):
                hs[i], cs[i] = cell(x, hs[i], cs[i])
                x = hs[i]
        return self.head(x)


K = 3

In [ ]:
from torch.utils.data import Dataset, DataLoader

PATCH, STRIDE = 64, 64
T = series.shape[0]
ser = torch.from_numpy(series)


def change_mask(ti):
    return (ser[ti, 0] > 0.5) & (ser[ti - 1, 0] < 0.5)


def index(targets, keep_empty=0.3):
    rng = np.random.default_rng(0)
    out = []
    for ti in targets:
        cm = change_mask(ti).numpy()
        for y in range(0, H - PATCH + 1, STRIDE):
            for x in range(0, W - PATCH + 1, STRIDE):
                if cm[y:y+PATCH, x:x+PATCH].any() or rng.random() < keep_empty:
                    out.append((ti, y, x))
    return out


class WinDS(Dataset):
    def __init__(self, idx):
        self.idx = idx

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, k):
        ti, y, x = self.idx[k]
        seq = ser[ti - K:ti, :, y:y+PATCH, x:x+PATCH]
        target = change_mask(ti)[y:y+PATCH, x:x+PATCH].float()[None]
        return seq, target


TRAIN_T = list(range(K, T - 1))
train_loader = DataLoader(WinDS(index(TRAIN_T)), batch_size=32, shuffle=True)
print("train targets", TRAIN_T, "patches", len(train_loader.dataset))

In [ ]:
def tversky_loss(logits, target, alpha=0.3, beta=0.7, gamma=0.75, eps=1.0):
    p = torch.sigmoid(logits)
    tp = (p * target).sum((1, 2, 3))
    fp = (p * (1 - target)).sum((1, 2, 3))
    fn = ((1 - p) * target).sum((1, 2, 3))
    index = (tp + eps) / (tp + alpha * fp + beta * fn + eps)
    return ((1 - index) ** gamma).mean()


def loss_fn(logits, target):
    weight = torch.tensor(10.0, device=logits.device)
    wbce = nn.functional.binary_cross_entropy_with_logits(logits, target, pos_weight=weight)
    return wbce + tversky_loss(logits, target)


model = ConvLSTMForecaster(C).to(DEV)
opt = torch.optim.Adam(model.parameters(), 1e-3)
x, y = next(iter(train_loader))
x, y = x.to(DEV), y.to(DEV)
for i in range(50):
    opt.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()
    opt.step()
    if i % 10 == 0 or i == 49:
        print(f"overfit {i:2d} loss {loss.item():.4f}")

In [ ]:
model = ConvLSTMForecaster(C).to(DEV)
opt = torch.optim.Adam(model.parameters(), 1e-3)
for epoch in range(1, 41):
    model.train()
    total = 0.0
    for x, y in train_loader:
        x, y = x.to(DEV), y.to(DEV)
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
        total += loss.item() * x.size(0)
    if epoch % 5 == 0 or epoch == 1:
        print(f"epoch {epoch:3d}  loss {total/len(train_loader.dataset):.4f}")
torch.save(model.state_dict(), "convlstm_forecast.pt")
print("saved convlstm_forecast.pt")

In [ ]:
import matplotlib.pyplot as plt


def predict_change(ti):
    with torch.no_grad():
        return torch.sigmoid(model(ser[ti - K:ti].unsqueeze(0).to(DEV)))[0, 0].cpu().numpy()


def fom(pred, true):
    hits = int((pred & true).sum())
    misses = int((~pred & true).sum())
    false_alarms = int((pred & ~true).sum())
    return hits / max(hits + misses + false_alarms, 1), hits, misses, false_alarms


prob = predict_change(T - 1)
b23, b24 = built_of(2023), built_of(2024)
new_true = (b24 > 0.5) & (b23 < 0.5)

best = (0, 0.5)
for thr in np.arange(0.1, 0.61, 0.05):
    score, *_ = fom((prob > thr) & (b23 < 0.5), new_true)
    if score > best[0]:
        best = (score, thr)
THR = best[1]
new_pred = (prob > THR) & (b23 < 0.5)
score, hits, misses, false_alarms = fom(new_pred, new_true)
persistence, *_ = fom(np.zeros_like(new_true), new_true)
print(f"ConvLSTM  Figure of Merit {score:.3f} @ thr {THR:.2f}  (hits {hits}, miss {misses}, false {false_alarms})")
print(f"persist.  Figure of Merit {persistence:.3f}")

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(b24, cmap="viridis")
ax[0].set_title("actual built 2024")
ax[1].imshow(prob, cmap="magma", vmin=0, vmax=1)
ax[1].set_title("predicted 2024 growth prob")
overlay = np.zeros((H, W, 3))
overlay[new_true & new_pred] = [1, 0, 0]
overlay[new_true & ~new_pred] = [0, 0, 1]
overlay[~new_true & new_pred] = [1, 1, 0]
ax[2].imshow(overlay)
ax[2].set_title("red hit / blue miss / yellow false")
for a in ax:
    a.axis("off")
fig.suptitle(f"Held-out 2024 growth -- Figure of Merit {score:.3f}")
fig.tight_layout()
fig.savefig("validation_2024.png", dpi=130)
plt.show()

In [ ]:
def rebuild(built, year):
    return channels_from_built(np.clip(built, 0, 1), year)


def save_tif(arr, path):
    profile = dict(driver="GTiff", height=H, width=W, count=1, dtype="float32",
                   crs=CRS, transform=TRANSFORM, nodata=0)
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(arr.astype("float32"), 1)


roll = [series[t] for t in range(T)]
built_now = built_of(2024).copy()
base24 = built_of(2024)
forecasts = {}
for year in range(2025, 2031):
    seq = torch.from_numpy(np.stack(roll[-K:]))[None].to(DEV)
    with torch.no_grad():
        change = torch.sigmoid(model(seq))[0, 0].cpu().numpy()
    built_now = np.maximum(built_now, (change > THR).astype("float32"))
    roll.append(rebuild(built_now, year))
    forecasts[year] = built_now.copy()

for year in (2027, 2030):
    save_tif(forecasts[year], f"forecast_convlstm_{year}.tif")
    new = (forecasts[year] > 0.5) & (base24 < 0.5)
    print(f"{year}: +{new.sum()*1e-4:.2f} km^2 new built-up")

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, year, image in zip(ax, [2024, 2027, 2030], [base24, forecasts[2027], forecasts[2030]]):
    a.imshow(image, cmap="magma", vmin=0, vmax=1)
    a.set_title(f"built {year}")
    a.axis("off")
fig.suptitle("ConvLSTM autoregressive forecast")
fig.tight_layout()
fig.savefig("forecast_rollout.png", dpi=130)
plt.show()

In [ ]:
import zipfile

with zipfile.ZipFile("forecast_outputs.zip", "w") as z:
    for f in glob.glob("forecast_convlstm_*.tif") + glob.glob("*.png") + ["convlstm_forecast.pt"]:
        if os.path.exists(f):
            z.write(f)
from google.colab import files as gfiles
gfiles.download("forecast_outputs.zip")